# 10 - Identifying Weakness: Logistic Attribution

Given a table of binary outcomes and factor levels, `evaluate.attribute` fits a logistic analysis of deviance per factor and corrects across the family of tests. Here one factor (clarity) drives failure.

In [ ]:
import sys, pathlib
import proofloop.suite as gt
cat = gt.Catalog.load()
print(cat.summary())

In [ ]:
import random
from proofloop.suite.evaluate import attribute
rng = random.Random(7)
suite = gt.resolve(cat, ['multi_hop'], ['clarity', 'entity_aliasing', 'reasoning_cue'], mode='embedded')
scns = gt.compose(suite, [gt.QAItem(qid=f's{i}', query='m', answer='ok') for i in range(6)],
                  n_runs=120, seed=7)
rows = [{'correct': 0 if rng.random() < (0.6 if s.factor_levels['clarity'] == 'Misleading' else 0.15) else 1,
         **{f'f_{k}': v for k, v in s.factor_levels.items()}} for s in scns]
tab = attribute(rows, suite.factor_names)
if isinstance(tab, dict) and tab.get('method') == 'weak_link':
    print('Open baseline: calibrated logistic attribution (+ FDR) requires the GMS backend.')
    print('First-error heuristic (weak_link) instead:',
          tab['counts'] or '(needs per-component trajectories, absent in these synthetic rows)')
else:
    for t in tab:
        print(f"{t['factor']:16s} p_adj={t['p_value_adj']:.4f} R2={t['pseudo_r2']:.3f} sig={t['significant_adj']}")

## Visualization

In [ ]:
import forgeloop.gms as gms
if isinstance(tab, list) and tab:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt, pathlib
    FIGDIR = pathlib.Path("figures"); FIGDIR.mkdir(parents=True, exist_ok=True)
    names = [t['factor'] for t in tab]
    r2 = [t['pseudo_r2'] for t in tab]
    colors = ['#c33' if t['significant_adj'] else '#999' for t in tab]
    fig, ax = plt.subplots(figsize=(4.6, 3))
    ax.bar(names, r2, color=colors)
    ax.set_ylabel('pseudo-$R^2$')
    ax.set_title('Factor effect (red = significant after FDR)')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
    fig.tight_layout(); fig.savefig(FIGDIR / 'fig_attribution.png', dpi=150)
    print('wrote', FIGDIR / 'fig_attribution.png')
else:
    print('Figure skipped: the attribution chart needs the GMS logistic table (open fallback returns weak_link).')

## Self-check

In [ ]:
import forgeloop.gms as gms
if isinstance(tab, list) and tab:
    clar = [t for t in tab if t['factor'] == 'clarity'][0]
    assert clar['significant_adj']                          # clarity drives failure
    print('OK - logistic attribution')
else:
    print('OK - open baseline (calibrated logistic attribution + FDR is a GMS feature)')